<a href="https://colab.research.google.com/github/dylankam/fyp-model1/blob/main/finetune/fyp_finetune.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Always execute this cell first**

In [ ]:
%%capture
# 1. Install the absolute latest versions of everything
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes

system_prompt = (
    "You are a Cartesian mapping model for a humanoid robot. Output strictly JSON with a 'keyframes' array.\n"
    "JSON FORMAT REQUIRED:\n"
    "{'keyframes': [{'time_fraction': float, 'use_hand': 'left'|'right'|'both', '[hand]_hand_pos': [x,y,z], '[hand]_orientation': str, '[hand]_fingers': str}], 'duration': float}\n"
    "RULES:\n"
    "- TIMING & HOLDING: time_fraction is a float (0.1 to 1.0). To hold a pose, hit the target fast (e.g., 0.3), then duplicate that exact position in a final keyframe at time_fraction=1.0.\n"
    "- ANCHORING: 'use_hand' dictates active limbs. If 'right', output ONLY right_hand keys. If 'left', output ONLY left_hand keys. If 'both', output both. Missing hands freeze in place.\n"
    "- COORDS: Normalized floats [-1.0 to 1.0]. X (0.0=torso, 1.0=max forward). Z (-1.0=waist, 0.0=center, 1.0=head).\n"
    "- Y-AXIS POLARITY (CRITICAL): The Y-axis represents left/right width.\n"
    "  * The 'left_hand_pos' Y-coordinate MUST ALWAYS be a positive number (0.0 to 1.0).\n"
    "  * The 'right_hand_pos' Y-coordinate MUST ALWAYS be a negative number (0.0 to -1.0).\n"
    "  * Example 1: left_hand_pos: [0.3, +0.4, 0.1] is GOOD  [0.3, -0.4, 0.1] is ILLEGAL . \n"
    "  * Example 2: right_hand_pos: [0.3, -0.4, 0.1] is GOOD [0.3, +0.4, 0.1] is ILLEGAL . \n"
    "- ORTHOGONALITY (CRITICAL ANATOMY): Palms and fingers CANNOT share an axis. \n"
    "  * If orientation is 'palms_forward' or 'palms_backward', fingers MUST be 'up', 'down', 'left', or 'right'.\n"
    "  * If orientation is 'palms_up' or 'palms_down', fingers MUST be 'forward', 'backward', 'left', or 'right'.\n"
    "  * If orientation is 'palms_in' or 'palms_out', fingers MUST be 'forward', 'backward', 'up', or 'down'.\n"
    "- COLLISION AVOIDANCE: Keep arms physically separated."
)

# For if you don't already have the Model Weights:

**Fine Tune the Model using uploaded training and validation datasets**

In [ ]:

from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template
import torch
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

# 2. Load the Base Model
max_seq_length = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
)

# 3. Configure the LoRA Adapter
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0.05,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

# 4. Format the Dataset Explicitly
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama-3",
    mapping = {"role" : "role", "content" : "content", "user" : "user", "assistant" : "assistant"}
)

def formatting_prompts_func(examples):
    convos = examples["messages"]
    # batched=True guarantees this is a safe list of lists
    for convo in convos:
        if convo[0]["role"] == "system":
            convo[0]["content"] = system_prompt  # Forces it to use the new prompt
    texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False) for convo in convos]
    return { "text" : texts }

print("Loading datasets...")
# 1. Load both files explicitly
train_data = load_dataset("json", data_files="dataset.jsonl", split="train")
eval_data = load_dataset("json", data_files="eval_dataset.jsonl", split="train") # Still split="train" because it's a single file

# 2. Format both datasets using your formatting function
train_set = train_data.map(formatting_prompts_func, batched = True, remove_columns=["messages"])
eval_set = eval_data.map(formatting_prompts_func, batched = True, remove_columns=["messages"])


print("\n DATASET VERIFICATION (Sample 0):")
print(train_set[0]["text"])
print("-----------------------------------\n")

# 5. Configure the Trainer (Modern API)
trainer = SFTTrainer(
    model = model,
    processing_class = tokenizer,
    train_dataset = train_set,
    eval_dataset = eval_set,
    args = SFTConfig(
        dataset_text_field = "text",
        max_seq_length = max_seq_length,
        packing = False,
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 3,
        learning_rate = 1e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        do_eval = True,               # Explicitly force evaluation on
        eval_strategy = "steps",
        eval_steps = 5,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "cosine",
        seed = 3407,
        output_dir = "outputs",
    ),
)

# 6. EXECUTE TRAINING
print("Starting Fine-Tune...")
trainer_stats = trainer.train()
print("\n Training Complete!")

# 7. Save the Adapter
model.save_pretrained("lora_model")
tokenizer.save_pretrained("lora_model")
print("Model saved to 'lora_model' folder.")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.5.8: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/220 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.26k [00:00<?, ?B/s]

...

Step,Training Loss,Validation Loss
5,1.572259,1.466836
10,1.136503,0.992646
15,0.575318,0.488961
20,0.242889,0.209713
25,0.173301,0.173277
30,0.160989,0.166422
35,0.144617,0.160013
40,0.150051,0.156942
45,0.144782,0.155560
50,0.138709,0.155254


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-51/tokenizer_config.json.



 Training Complete!


Unsloth: Restored added_tokens_decoder metadata in lora_model/tokenizer_config.json.


Model saved to 'lora_model' folder.


In [ ]:
# CHECK OUTPUT FORMAT IS CORRECT
# Enable native 2x faster inference
FastLanguageModel.for_inference(model)


# 2. Create a brand new test scenario the model has never seen
test_messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": '{"text": "Look at the tiny bug on the floor!", "use_hand": "right", "description": "The robot points its right hand sharply down towards the ground near its feet.", "duration": 2.0}'}
]

# 3. Format using the chat template
inputs = tokenizer.apply_chat_template(
    test_messages,
    tokenize=True,
    add_generation_prompt=True, # Crucial: Tells Llama it's time to act as the assistant
    return_tensors="pt"
).to("cuda")

# 4. Generate the payload
print("Generating JSON payload...")
outputs = model.generate(input_ids=inputs, max_new_tokens=512, use_cache=True, temperature=0.1)

# 5. Decode and clean the output
decoded_output = tokenizer.batch_decode(outputs)[0]

# Extract just the assistant's JSON response
final_json = decoded_output.split("<|start_header_id|>assistant<|end_header_id|>")[-1].replace("<|eot_id|>", "").strip()

print("\n=== MODEL OUTPUT ===")
print(final_json)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Generating JSON payload...


Both `max_new_tokens` (=512) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)



=== MODEL OUTPUT ===
{'keyframes': [{'time_fraction': 0.4, 'use_hand': 'right', 'right_hand_pos': [0.1, -0.2, -0.8], 'right_orientation': 'palms_down', 'right_fingers': 'forward'}, {'time_fraction': 1.0, 'use_hand': 'right', 'right_hand_pos': [0.1, -0.2, -0.8], 'right_orientation': 'palms_down', 'right_fingers': 'forward'}], 'duration': 2.0}


**Start ngrok server with an newly created model weights:**

In [ ]:
!pip install pyngrok flask

from flask import Flask, request, jsonify
from pyngrok import ngrok
import json
from google.colab import userdata

# 1. Securely Authenticate your tunnel
try:
    my_ngrok_token = userdata.get('NGROK_TOKEN')
    ngrok.set_auth_token(my_ngrok_token)
except Exception as e:
    print("Error: Could not find NGROK_TOKEN in Colab Secrets!")
    print(e)

app = Flask(__name__)

@app.route('/generate_gesture', methods=['POST'])
def generate_gesture():
    # Receive the LLM1 output from your laptop
    incoming_data = request.json
    user_content = json.dumps(incoming_data)

    # Format the prompt using the model's native template
    messages = [
        {"role": "system", "content": system_prompt}, # Make sure system_prompt is defined above!
        {"role": "user", "content": user_content}
    ]
    inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")

    # Generate the gesture
    outputs = model.generate(input_ids=inputs, max_new_tokens=512, use_cache=True, temperature=0.1)
    decoded = tokenizer.batch_decode(outputs)[0]
    final_json_str = decoded.split("<|start_header_id|>assistant<|end_header_id|>")[-1].replace("<|eot_id|>", "").strip()

    # Send it back
    return jsonify({"gesture_payload": final_json_str})

# 2. Open the tunnel and start the server
public_url = ngrok.connect(5000).public_url
print(f"\n YOUR API URL IS: {public_url}/generate_gesture \n")

# This will block the cell and keep the server running continuously
app.run(port=5000)


 YOUR API URL IS: https://cortex-thermal-lurk.ngrok-free.dev/generate_gesture 

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=512) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warni

Zip weights for easy download:

In [ ]:
!zip -r lora_model_backup.zip lora_model

  adding: lora_model/ (stored 0%)
  adding: lora_model/tokenizer.json (deflated 85%)
  adding: lora_model/tokenizer_config.json (deflated 96%)
  adding: lora_model/adapter_config.json (deflated 59%)
  adding: lora_model/adapter_model.safetensors (deflated 8%)
  adding: lora_model/chat_template.jinja (deflated 67%)
  adding: lora_model/README.md (deflated 65%)


# For if you already have Model Weights:

**Run This Cell to start ngrok server with a model zip file named lora_model_backup.zip**

In [ ]:
from google.colab import files

print("Please select your lora_model_backup.zip file...")
uploaded = files.upload()

# Once the upload hits 100%, run the unzip command
!unzip -q /content/lora_model_backup.zip -d /content/lora_adapter

Please select your lora_model_backup.zip file...


Saving lora_model_backup.zip to lora_model_backup (1).zip
replace /content/lora_adapter/lora_model/tokenizer.json? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace /content/lora_adapter/lora_model/tokenizer_config.json? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace /content/lora_adapter/lora_model/adapter_config.json? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace /content/lora_adapter/lora_model/adapter_model.safetensors? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace /content/lora_adapter/lora_model/chat_template.jinja? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace /content/lora_adapter/lora_model/README.md? [y]es, [n]o, [A]ll, [N]one, [r]ename: n


In [ ]:
# 1. Install dependencies
!pip install -q pyngrok flask
!unzip -q /content/lora_model_backup.zip -d /content/lora_adapter
from flask import Flask, request, jsonify
from pyngrok import ngrok
import json
import torch
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template
from google.colab import userdata

# ==========================================
# 2. LOAD MODEL VIA UNSLOTH
# ==========================================
max_seq_length = 2048

print("Loading model and LoRA adapter via Unsloth...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "/content/lora_adapter/lora_model", # Point to the model
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
)


# Apply the exact chat template mapping used during training
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama-3",
    mapping = {"role" : "role", "content" : "content", "user" : "user", "assistant" : "assistant"}
)

# Optimizes the model for 2x faster generation
FastLanguageModel.for_inference(model)
print("Model loaded and optimized for inference")

# ==========================================
# 3. SERVER SETUP
# ==========================================
try:
    my_ngrok_token = userdata.get('NGROK_TOKEN')
    ngrok.set_auth_token(my_ngrok_token)
except Exception as e:
    print("Error: Could not find NGROK_TOKEN in Colab Secrets")
    print(e)

app = Flask(__name__)

@app.route('/generate_gesture', methods=['POST'])
def generate_gesture():
    incoming_data = request.json
    user_content = json.dumps(incoming_data)


    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_content}
    ]

    # Format using the exact template defined above
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to("cuda")

    # Generate the gesture
    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=512,
        use_cache=True,
        temperature=0.1
    )

    decoded = tokenizer.batch_decode(outputs)[0]
    final_json_str = decoded.split("<|start_header_id|>assistant<|end_header_id|>")[-1].replace("<|eot_id|>", "").strip()

    return jsonify({"gesture_payload": final_json_str})

# 4. Open the tunnel and start the server
ngrok.kill()
public_url = ngrok.connect(5000).public_url
print(f"\n YOUR API URL IS: {public_url}/generate_gesture \n")

app.run(port=5000)

replace /content/lora_adapter/lora_model/tokenizer.json? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace /content/lora_adapter/lora_model/tokenizer_config.json? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace /content/lora_adapter/lora_model/adapter_config.json? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace /content/lora_adapter/lora_model/adapter_model.safetensors? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace /content/lora_adapter/lora_model/chat_template.jinja? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace /content/lora_adapter/lora_model/README.md? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
Loading model and LoRA adapter via Unsloth...
==((====))==  Unsloth 2026.6.1: Fast Llama patching. Transformers: 5.5.0.
   \\   /| 

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/220 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/345 [00:00<?, ?B/s]

Unsloth: Will load unsloth/llama-3-8b-instruct as a legacy tokenizer.
Unsloth 2026.6.1 patched 32 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


Model loaded and optimized for inference

 YOUR API URL IS: https://cortex-thermal-lurk.ngrok-free.dev/generate_gesture 

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=512) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warni